<div dir="rtl">

# 🎬 AI Video Analyzer — نسخة كولاب

التفريغ الصوتي والقص والتصدير بيتعمل هنا على كولاب (بكارت شاشة مجاني)، والواجهة بتتفتح في تاب منفصل في المتصفح.
الفيديوهات بتتقرا من جوجل درايف والمقاطع بتتحفظ عليه مباشرة — جهازك بيعرض الواجهة بس.

## أول مرة بس
1. **مفتاح Gemini:** اضغط أيقونة المفتاح 🔑 (Secrets) في الشريط الجانبي ← **Add new secret** ← الاسم `GEMINI_API_KEY` والقيمة مفتاحك من [Google AI Studio](https://aistudio.google.com/app/apikey) ← فعّل **Notebook access**.
   - اختياري: `OPENROUTER_API_KEY` لو عايز تستخدم OpenRouter في التحليل.
2. **كارت الشاشة:** النوت بوك بيطلب T4 GPU تلقائيًا. لو ظهر إنه من غير GPU: **Runtime ← Change runtime type ← T4 GPU ← Save**.

## كل مرة
1. **Runtime ← Run all** (أو `Ctrl+F9`).
2. وافق على صلاحية الوصول لجوجل درايف.
3. استنى لحد ما يظهر رابط **«🚀 افتح الواجهة»** تحت، واضغط عليه.

> خلي تاب كولاب ده مفتوح طول ما انت شغال. الجلسة المجانية بتقفل لو فضلت من غير استخدام فترة، وأقصاها حوالي 12 ساعة —
> كل المشاريع والقوالب محفوظة على الدرايف، فتقدر تكمل من مكانك بتشغيل النوت بوك تاني.

</div>

In [ ]:
#@title ⚙️ الإعدادات { display-mode: "form" }
#@markdown اسم الفولدر على جوجل درايف اللي هيتحفظ فيه كل حاجة (المشاريع، القوالب، المقاطع المصدّرة):
DRIVE_FOLDER = "AIVideoAnalyzer"  #@param {type:"string"}
PORT = 8000  #@param {type:"integer"}

In [ ]:
#@title 📦 تثبيت المكتبات (حوالي دقيقة في أول كل جلسة)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "faster-whisper>=1.1", "fastapi>=0.110", "uvicorn>=0.29"], check=True)
print("✓ تم تثبيت المكتبات")

In [ ]:
#@title 🧩 تجهيز ملفات التطبيق
import base64, io, shutil, zipfile
APP_DIR = "/content/ava_app"
APP_ZIP = """
...
"""
shutil.rmtree(APP_DIR, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(APP_ZIP.split())))) as z:
    z.extractall(APP_DIR)
print("✓ ملفات التطبيق جاهزة")

In [ ]:
#@title 🚀 تشغيل التطبيق
import json, os, signal, socket, subprocess, sys, time, urllib.request
from google.colab import drive, output, userdata

APP_DIR = "/content/ava_app"
LOG_PATH = "/content/ava_server.log"
PID_PATH = "/content/ava_server.pid"
drive.mount("/content/drive")

def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:  # secret missing, or notebook access not granted
        return ""

env = dict(os.environ,
           AVA_MOUNT_ROOT="/content/drive/MyDrive",
           AVA_DATA_DIR=DRIVE_FOLDER.strip() or "AIVideoAnalyzer",
           AVA_WORK_DIR="/content/ava_work",
           GEMINI_API_KEY=secret("GEMINI_API_KEY"),
           OPENROUTER_API_KEY=secret("OPENROUTER_API_KEY"),
           PYTHONUNBUFFERED="1")

# Stop a server left over from a previous run of this cell. The PID lives in a file
# (not a variable) so this still works after a kernel restart; it's checked against
# /proc first because PIDs get reused.
try:
    old_pid = int(open(PID_PATH).read().strip())
    if b"ava.server:app" in open(f"/proc/{old_pid}/cmdline", "rb").read():
        os.kill(old_pid, signal.SIGTERM)
except (OSError, ValueError):
    pass
for _ in range(40):  # wait for the port to be released
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", PORT)) != 0:
            break
    time.sleep(0.25)

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "ava.server:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=APP_DIR, env=env, stdout=open(LOG_PATH, "w"), stderr=subprocess.STDOUT)
with open(PID_PATH, "w") as f:
    f.write(str(server.pid))

status = None
for _ in range(90):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/status", timeout=3) as r:
            status = json.load(r)
        break
    except Exception:
        time.sleep(1)

if status is None:
    print(open(LOG_PATH, encoding="utf-8", errors="replace").read()[-4000:])
    raise RuntimeError("السيرفر ما اشتغلش — شوف السجل اللي فوق")

print("✓ السيرفر شغال")
print("• كارت الشاشة:", status["gpu"] or "⚠ مش متاح (التفريغ هيبقى أبطأ) — Runtime ← Change runtime type ← T4 GPU")
print("• التصدير:", "بكارت الشاشة (NVENC)" if status["nvenc"] else "بالمعالج")
print("• Gemini:", "✓" if status["gemini_key"] else "✗ ضيف GEMINI_API_KEY في Secrets وشغّل الخلية دي تاني")
print("• الفولدر على الدرايف:", "My Drive/" + status["data_root"])
print()
try:
    output.serve_kernel_port_as_window(PORT, anchor_text="🚀 افتح الواجهة")
except TypeError:  # older google.colab without anchor_text
    output.serve_kernel_port_as_window(PORT)

In [ ]:
#@title 📜 سجل السيرفر (لو حصلت مشكلة، شغّل الخلية دي وابعت اللي يظهر)
print(open("/content/ava_server.log", encoding="utf-8", errors="replace").read()[-8000:])